# SpendLens: pretrained models → fine-tuning → evaluation → Hugging Face
Use a Colab GPU runtime. Training begins only when you execute the training cells. Demo data is synthetic. The two checkpoints are separate.


In [ ]:
!git clone https://github.com/zaynfuture/isom-group-project.git /content/isom-group-project
%cd /content/isom-group-project
%pip install -r requirements.txt


## Inspect transactions
The reviewed Citi mapping currently covers five MCCs. Other codes stay unmapped.


In [ ]:
import sys
sys.path.insert(0, 'src')
from isom_project.spending import demo_transactions, monthly_examples
data = demo_transactions()
display(data)
display(monthly_examples(data))


## Fine-tune both tasks from Hugging Face pretrained DistilBERT
Merchant labels are derived from known MCCs; model inputs exclude MCC. Forecast inputs contain two prior months, never the future target month.


In [ ]:
!python scripts/train_spendlens.py --task merchant --epochs 3


In [ ]:
!python scripts/train_spendlens.py --task forecast --epochs 3


## Review evaluation before upload
Compare each test Macro-F1 against its baseline. Train scores are in-sample; synthetic results require external validation.


In [ ]:
import json
from pathlib import Path
for task in ['merchant', 'forecast']:
    print(task)
    display(json.loads(Path(f'artifacts/spendlens_{task}/evaluation.json').read_text()))


## Upload model files to zhengzhihust
Enable explicitly after reviewing evaluation. Authentication is interactive; raw transactions are excluded.


In [ ]:
PUBLISH = False
if PUBLISH:
    from huggingface_hub import notebook_login
    import subprocess
    notebook_login()
    for task in ['merchant', 'forecast']:
        subprocess.run([sys.executable, 'scripts/upload_spendlens.py', '--task', task], check=True)


## Configure Streamlit after successful uploads
Use these root-level secrets:
```toml
SPENDLENS_MERCHANT_MODEL = "zhengzhihust/spendlens-merchant-classifier"
SPENDLENS_FORECAST_MODEL = "zhengzhihust/spendlens-spending-forecast"
```
Application code stays in GitHub. Models, tokenizers and evaluation evidence stay in Hugging Face. Verify both inference paths after deployment.
